### Dependencies

In [ ]:
import datetime
import importlib
import sys

# Ignore Warning
import warnings

import dask.dataframe as dd
import holoviews as hv
import numpy as np
import pandas as pd
from holoviews.element.tiles import OSM

warnings.filterwarnings("ignore")

In [ ]:
def reload_package(package):
    for module in list(sys.modules.keys()):
        if module.startswith(package.__name__):
            importlib.reload(sys.modules[module])

In [ ]:
import moveminer2
from moveminer2.core.trajectory import Trajectory
from moveminer2.utils.config import col_names

reload_package(moveminer2)
analyzer = moveminer2.Analyzer()
# Visualizer
kde_plot = analyzer.kde_plot
spatial_plot = analyzer.spatial_plot
heatmap_plot = analyzer.heatmap_plot
polar_plot = analyzer.polar_plot
# Metrics
distance_calculator = analyzer.metrics.distance_calculator
timediff_calculator = analyzer.metrics.timediff_calculator
speed_calculator = analyzer.metrics.speed_calculator
radius_calculator = analyzer.metrics.radius_calculator
turning_angle_calculator = analyzer.metrics.turning_angle_calculator
od_matrix_calculator = analyzer.metrics.od_matrix_calculator
# Preprocessing
stop_detector = analyzer.preprocessing.stop_detector
outlier_detector = analyzer.preprocessing.outlier_detector
cluster_detector = analyzer.preprocessing.cluster_detector
compressor = analyzer.preprocessing.compressor
segmenter = analyzer.preprocessing.segmenter

### Setup

In [ ]:
def load_InTAS(path: str) -> pd.DataFrame:
    intas = dd.read_csv(
        path,
        sep=";",
        usecols=[
            "timestep_time",
            "vehicle_id",
            "vehicle_x",
            "vehicle_y",
            "person_id",
            "person_x",
            "person_y",
        ],
    )
    intas[col_names.TRAJECTORY_ID] = intas["vehicle_id"].fillna(
        intas["person_id"]
    )
    intas[col_names.X] = intas["vehicle_x"].fillna(intas["person_x"])
    intas[col_names.Y] = intas["vehicle_y"].fillna(intas["person_y"])
    InTAS_datetime = datetime.datetime(2019, 11, 1)
    intas["t"] = dd.to_datetime(
        intas["timestep_time"],
        unit="s",
        origin=InTAS_datetime,
    )
    return intas.compute()

In [ ]:
# Load datasets
intas1 = load_InTAS("./datasets/6_as_10hrs/fcd.csv")
intas2 = load_InTAS("./datasets/15_as_19hrs/fcd.csv")

In [ ]:
intas_title1 = "InTAS (6 to 10 hours)"
intas_mt1 = Trajectory(intas1)

In [ ]:
intas_title2 = "InTAS (15 to 19 hours)"
intas_mt2 = Trajectory(intas2)

In [ ]:
tiles = OSM().opts(
    alpha=0.5,
    bgcolor="black",
)

In [ ]:
plot_width = 960
plot_height = 540

In [ ]:
output_folder = "./assets/MoveMiner"

### Plot Points / Spatial Projection ✅

In [ ]:
hv.extension("bokeh")

In [ ]:
intas_plot1 = tiles * spatial_plot(
    intas_mt1,
    title=intas_title1,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)
intas_plot2 = tiles * spatial_plot(
    intas_mt2,
    title=intas_title2,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)
hv.save(
    intas_plot1 + intas_plot2,
    f"{output_folder}/Spatial Projection/intas1_2.png",
    fmt="png",
)

intas_plot1 + intas_plot2

### Turning Angles (Point-to-Point) ✅

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
%%time
intas_mt1 = turning_angle_calculator.add_turning_angles(intas_mt1)
intas_mt2 = turning_angle_calculator.add_turning_angles(intas_mt2)

In [ ]:
column = col_names.TURNING_ANGLE

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
# from matplotlib.ticker import FuncFormatter

# opts = {
#     "xlabel": "Turning Angles (º)",
#     "ylabel": "",
#     "yticks": 5,
#     "yformatter": FuncFormatter(lambda x, _: f"{x / 24000000}"),
#     # "padding": 100,
#     "fig_inches": (6, 6),
# }
# opts1 = {"title": intas_title1, **opts}
# opts2 = {"title": intas_title2, **opts}
# hist_plot(np.radians(intas_mt1[column]), bins=360).options(
#     backend="matplotlib", projection="polar", **opts1
# ) + (
#     hist_plot(np.radians(intas_mt2[column]), bins=360).options(
#         backend="matplotlib", projection="polar", **opts2
#     )
# )

In [ ]:
opts = {
    "xlabel": "Turning Angles (º)",
    "ylabel": "Frequency",
}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = (intas_mt1[column].hvplot.hist(**opts1)) + (
    intas_mt1[column].hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots,
    f"{output_folder}/Turning Angles (Point-to-Point)/intas1_2.pdf",
    dpi=300,
)

### Distance (Point-to-Point) ✅

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_mt1 = distance_calculator.add_distance_column(intas_mt1)
intas_mt2 = distance_calculator.add_distance_column(intas_mt2)

In [ ]:
column = col_names.DISTANCE

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")
opts = {"xlabel": "Distance (m)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}

In [ ]:
hist_plots = intas_mt1[column].hvplot.hist(**opts1) + (
    intas_mt2[column].hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots,
    f"{output_folder}/Distance (Point-to-Point)/intas1_2.pdf",
    dpi=300,
)

In [ ]:
hist_plots = intas_mt1.groupby(col_names.TRAJECTORY_ID)[
    column
].max().hvplot.hist(**opts1) + (
    intas_mt2.groupby(col_names.TRAJECTORY_ID)[column]
    .max()
    .hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots,
    f"{output_folder}/Distance (Point-to-Point)/max_intas1_2.pdf",
    dpi=300,
)

### Total Distance Displacement ✅

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas1_total_distance = distance_calculator.calculate_total_distance(intas_mt1)
intas2_total_distance = distance_calculator.calculate_total_distance(intas_mt2)

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
opts = {"xlabel": "Total Distance (m)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = intas1_total_distance.reset_index().hvplot.hist(**opts1) + (
    intas2_total_distance.reset_index().hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(hist_plots, f"{output_folder}/Total Distance/intas1_2.pdf", dpi=300)

### Straight Line Distance

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
straight_line_distance1 = distance_calculator.calculate_straight_line_distance(
    intas_mt1
)
straight_line_distance2 = distance_calculator.calculate_straight_line_distance(
    intas_mt2
)

In [ ]:
column = col_names.STRAIGHT_LINE_DISTANCE

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
opts = {"xlabel": "Straight Line Distance (m)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = (straight_line_distance1[column].hvplot.hist(**opts)) + (
    straight_line_distance2[column].hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots, f"{output_folder}/Straight Line Distance/intas1_2.pdf", dpi=300
)

### Waiting Times (Point-to-Point) ✅

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_mt1 = timediff_calculator.add_timediff_column(intas_mt1)
intas_mt2 = timediff_calculator.add_timediff_column(intas_mt2)

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
column = col_names.TIME_DIFF

In [ ]:
opts = {"xlabel": "Waiting Times (s)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = (intas_mt1[column].hvplot.hist(**opts1)) + (
    intas_mt2[column].hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots,
    f"{output_folder}/Waiting Time (Point-to-Point)/intas1_2.pdf",
    dpi=300,
)

### Speed (Point-to-Point) ✅

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_mt1 = speed_calculator.add_speed_column(intas_mt1)
intas_mt2 = speed_calculator.add_speed_column(intas_mt2)

In [ ]:
column = col_names.SPEED

In [ ]:
opts = {"xlabel": "Speed (m/s)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = (intas_mt1[column].hvplot.hist(**opts1)) + (
    intas_mt2[column].hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots, f"{output_folder}/Speed (Point-to-Point)/intas1_2.pdf", dpi=300
)

### Stay Locations ✅⏳

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_mt1 = stop_detector.add_stop_column(intas_mt1)
intas_mt2 = stop_detector.add_stop_column(intas_mt2)

In [ ]:
column = col_names.STOP

In [ ]:
print(f"{intas_mt1[column].sum()} stay points found in {intas_title1}")
print(f"{intas_mt2[column].sum()} stay points found in {intas_title2}")

In [ ]:
hv.extension("bokeh")
intas_plot1 = tiles * spatial_plot(
    intas_mt1[intas_mt1[column]],
    title=intas_title1,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)
intas_plot2 = tiles * spatial_plot(
    intas_mt2[intas_mt2[column]],
    title=intas_title2,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)

layout = intas_plot1 + intas_plot2
hv.save(layout, f"{output_folder}/Stay Locations/intas1_2.png")
layout

### Radius of Gyration ✅⏳

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_radius_gyration1 = radius_calculator.calculate_radius_of_gyration(
    intas_mt1
)
intas_radius_gyration2 = radius_calculator.calculate_radius_of_gyration(
    intas_mt2
)

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
opts = {"xlabel": "Radius of Gyration (m)", "ylabel": "Frequency"}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
hist_plots = (intas_radius_gyration1.reset_index().hvplot.hist(**opts1)) + (
    intas_radius_gyration2.reset_index().hvplot.hist(**opts2)
)
hist_plots

In [ ]:
hv.save(
    hist_plots, f"{output_folder}/Radius of Gyration/intas1_2.pdf", dpi=300
)

### Outliers Detection ✅⏳

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
max_speed_ms = 138.888889  # 500 km/h

In [ ]:
intas_mt1 = outlier_detector.detect_outliers(intas_mt1, max_speed_ms)
intas_mt2 = outlier_detector.detect_outliers(intas_mt2, max_speed_ms)

In [ ]:
column = col_names.OUTLIER

In [ ]:
print(
    f"{len(intas_mt1[intas_mt1[column]])} outliers with speed above {max_speed_ms} m/s in {intas_title1}"
)
print(
    f"{len(intas_mt2[intas_mt2[column]])} outliers with speed above {max_speed_ms} m/s in {intas_title2}"
)

In [ ]:
hv.extension("bokeh")
intas_plot1 = tiles * spatial_plot(
    intas_mt1[~intas_mt1[column]],
    title=intas_title1,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)
intas_plot2 = tiles * spatial_plot(
    intas_mt2[~intas_mt2[column]],
    title=intas_title2,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)

layout = intas_plot1 + intas_plot2
hv.save(layout, f"{output_folder}/Outliers Detection/intas1_2.png")
layout

### Clustering ✅⏳

In [ ]:
intas_mt1["dataset_id"] = 1
intas_mt2["dataset_id"] = 2

In [ ]:
intas = pd.concat([intas_mt1, intas_mt2])

In [ ]:
intas = cluster_detector.detect_clusters(intas)

In [ ]:
intas_mt1 = intas[intas["dataset_id"] == 1]
intas_mt2 = intas[intas["dataset_id"] == 2]

In [ ]:
column = col_names.CLUSTER

In [ ]:
from bokeh.models import BasicTicker, PrintfTickFormatter

colorbar_opts = {
    "ticker": BasicTicker(),
    "formatter": PrintfTickFormatter(format="%d"),
}
hv.extension("bokeh")
intas_plot1 = tiles * spatial_plot(
    intas_mt1,
    color=column,
    title=intas_title1,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="Set3",
    width=plot_width,
    height=plot_height,
).opts(
    colorbar_opts=colorbar_opts,
)
intas_plot2 = tiles * spatial_plot(
    intas_mt2,
    color=column,
    title=intas_title2,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="Set3",
    width=plot_width,
    height=plot_height,
).opts(
    colorbar_opts=colorbar_opts,
)
layout = intas_plot1 + intas_plot2
hv.save(intas_plot1, f"{output_folder}/Clustering/intas1.png")
hv.save(intas_plot2, f"{output_folder}/Clustering/intas2.png")
layout

### Origin-Destination Matrix ✅⏳

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_od1 = od_matrix_calculator.calculate_od_matrix(intas_mt1)
intas_od2 = od_matrix_calculator.calculate_od_matrix(intas_mt2)

In [ ]:
hv.extension("matplotlib")
hv.output(fig="svg")

In [ ]:
opts = {
    "xlabel": "Origin",
    "ylabel": "Destination",
    "colorbar": True,
    "cmap": "summer",
}
opts1 = {"title": intas_title1, **opts}
opts2 = {"title": intas_title2, **opts}
layout = heatmap_plot(intas_od1, **opts1) + heatmap_plot(intas_od2, **opts2)
hv.save(
    layout, f"{output_folder}/Origin-Destination Matrix/intas1_2.pdf", dpi=300
)
layout

### Compression ✅⏳

In [ ]:
intas_mt1 = intas_mt1.sort_values([col_names.TRAJECTORY_ID, col_names.T])
intas_mt2 = intas_mt2.sort_values([col_names.TRAJECTORY_ID, col_names.T])

In [ ]:
intas_compress1 = compressor.compress_trajectory(intas_mt1, max_distance_m=200)
intas_compress2 = compressor.compress_trajectory(intas_mt2, max_distance_m=200)

In [ ]:
hv.extension("bokeh")

In [ ]:
intas_plot1 = tiles * spatial_plot(
    intas_compress1,
    title=intas_title1,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)
intas_plot2 = tiles * spatial_plot(
    intas_compress2,
    title=intas_title2,
    rasterize=True,
    dynspread=True,
    colorbar=True,
    cnorm="eq_hist",
    cmap="inferno",
    width=plot_width,
    height=plot_height,
)

layout = intas_plot1 + intas_plot2
hv.save(layout, f"{output_folder}/Compression/intas1_2.png")
layout